## CHARGEMENT DU STAGING VERS LE BRONZE

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://192.168.1.230:30604/api/v1" 

#---------------------------------------------------------------------------------

#LORSQU'ON TRAVAILLE SUR JHUB
MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1" 

#---------------------------------------------------------------------------------

conf = (
    SparkConf()
    .setAppName("Job_Scrapping")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.extraJavaOptions", "-Dorg.apache.poi.util.IOUtils.setByteArrayMaxOverride=1000000000")
    .set("spark.driver.memory", "16g")
    # AJOUT DES PACKAGES : On ajoute le jar nessie-spark-extensions
    .set("spark.jars.packages", 
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,""com.crealytics:spark-excel_2.12:3.5.0_0.20.3")
     
     .set("spark.sql.extensions", 
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    
    # On définit le bucket bronze comme entrepôt par défaut du catalogue
    .set("spark.sql.catalog.nessie.warehouse", "s3a://bronze/")
    
    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [2]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

26/02/17 10:19:09 WARN Utils: Your hostname, Jarvis.local resolves to a loopback address: 127.0.0.1; using 172.16.30.46 instead (on interface en0)
26/02/17 10:19:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/datalab/.ivy2/cache
The jars for the packages stored in: /Users/datalab/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
com.crealytics#spark-excel_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0c619236-f7b0-4cf7-aae5-f23029bc8c8c;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central


:: loading settings :: url = jar:file:/Users/datalab/spark_env/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.poi#poi-ooxml;5.2.5 in central
	found org.apache.poi#poi-ooxml-lite;5.2.5 in central
	found org.apache.xmlbeans#xmlbeans;5.2.0 in central
	found org.apache.commons#commons-compress;1.25.0 in central
	found com.github.virtuald#curvesapi;1.08 in central
	found com.norbitltd#spoiwo_2.12;2.2.1 in central
	found com.github.tototoshi#scala-csv_2.12;1.3.10 in central
	found com.github.pjfanning#excel-streaming-reader;4.2.1 in central
	found com.github.pjfanning#poi-shared-strings;2.7.1 in central
	found org.slf4j#slf4j-api;2.0.9 in central
	found com.h2database#h2;2.2.224 in central
	found org.apache.commons#commons-text;1.11.0 in central
	found org.apache.commons#commons-lang3;3.13.0 in central
	found commons-io#commons-io;2.15.1 in central
	found org.apache.logging.log4j#log4j-api;2.22.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.logging.log4j#log4j-core;2.22.0 in central
:: resolution report :: resolve 222ms :

In [ ]:
path_minio = "<<CHEMIN VERS LE FICHIER SUR MINIO>>"

df_fusionne = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(path_minio)

26/02/17 10:19:24 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


## ON EFFECTUE LES TRAITEMENTS PERSONNALISES ICI

In [ ]:
table_target = "nessie.bronze.<<nom de la table>>"

# Créer le namespace dans Nessie s'il n'existe pas encore
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

df_fusionne.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(table_target)